# Beadform Qwen 2B on free Colab

Use only Colab's free runtime. Select a GPU manually only when it is offered without an upgrade prompt. If no free GPU is available, stop and retry another day. This notebook never buys compute or provisions another service.

Google says free runtimes can run for up to 12 hours depending on availability and usage; GPU type and actual lifetime vary. Keep the notebook interactive, save checkpoints to Drive, and resume in a later eligible session. Use one account and do not use keep-alives or other workarounds to bypass limits.

Before training, prepare a review-queue JSONL containing only data you have permission to use, with corrected scenes that you reviewed and validated with Beadform's compiler. The 21-row candidates file is not used automatically. The reviewed `training/approved-seed.jsonl` contains 20 text-only examples; fish is excluded. Twenty examples remain a small fine-tuning dataset. The exploratory prompts below are not training labels or evaluation results.


In [1]:
# Stop before installing packages or loading weights if this session has no free CUDA GPU.
import subprocess
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU was offered. Stop; do not select a paid runtime. Retry later with this same account.")
gpu_name = torch.cuda.get_device_name(0)
print("CUDA GPU:", gpu_name, "| PyTorch:", torch.__version__)
subprocess.run(["nvidia-smi"], check=False)


CUDA GPU: Tesla T4 | PyTorch: 2.11.0+cu128


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [3]:
# Mount your own Drive. This prompts you to authorize access.
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Beadform-free-qwen")
DATA_ROOT = DRIVE_ROOT / "data"
RUN_ID = "pilot-01"  # Change for a new experiment.
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
print("Drive run folder:", RUN_DIR)


Mounted at /content/drive
Drive run folder: /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01


In [4]:
# Clone the public plan branch. Keep Colab's own PyTorch/CUDA build.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/c-tustin/beadform.git"
REPO_REF = "runpod-training-guide"
REPO_DIR = Path("/content/beadform")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
with (RUN_DIR / "install.log").open("w") as log:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "training/requirements.txt"],
                   stdout=log, stderr=subprocess.STDOUT, check=True)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, text=True, capture_output=True)
(RUN_DIR / "pip-freeze.txt").write_text(freeze.stdout)
print("Dependencies installed; package versions saved to Drive.")


Dependencies installed; package versions saved to Drive.


## Optional base-model preview

Run once to see a few exploratory outputs before preparing training data. These prompts are not held-out evaluation examples and are not approved training targets. The model cache stays on Colab's temporary VM; preview results are saved to Drive.


In [ ]:
# Reruns keep the existing preview file.
import gc
import json
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
preview_path = RUN_DIR / "exploratory-preview.jsonl"
if preview_path.exists():
    print("Keeping existing preview:", preview_path)
else:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, device_map="auto", torch_dtype=dtype,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype))
    system = (REPO_DIR / "training" / "system.txt").read_text()
    prompts = [
        "A tiny frog with a rounded body, short legs, and two dark eyes.",
        "A butterfly with four broad, simple wings and a narrow body.",
        "A small teapot with a round body, a curved handle, and a short spout."
    ]
    with preview_path.open("w") as out:
        for prompt in prompts:
            messages = [{"role": "system", "content": system},
                        {"role": "user", "content": [{"type": "text", "text": prompt}]}]
            inputs = processor.apply_chat_template(messages, tokenize=True, return_dict=True,
                return_tensors="pt", add_generation_prompt=True)
            inputs = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in inputs.items()}
            with torch.inference_mode():
                tokens = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
            raw = processor.batch_decode(tokens[:, inputs["input_ids"].shape[-1]:],
                                         skip_special_tokens=True)[0]
            out.write(json.dumps({"model": MODEL_ID, "prompt": prompt, "output": raw}) + "\n")
            print("\nPROMPT:", prompt, "\nOUTPUT:\n", raw[:4000])
            del inputs, tokens
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Preview saved:", preview_path)


## Prepare approved data, evaluate, and train

If you have a larger approved review queue, put it at Drive's data/review-queue.jsonl. Otherwise this cell copies the 20-row approved seed from the repository to Drive. Twenty examples are still a small fine-tuning dataset; add more examples if possible. Image paths should be relative to the JSONL and point to original input images on Drive. Do not substitute finished-build photos. Choose at least two whole design families for held-out evaluation. You can stop after the optional preview.

The split and train check run before loading training weights. Base and tuned predictions use the same held-out rows. Checkpoints are written to Drive after each optimizer step so a later free Colab session can resume.


In [5]:
import hashlib
import json
import shutil
import subprocess
import sys

REVIEW_QUEUE = DATA_ROOT / "review-queue.jsonl"
HOLDOUT_FAMILIES = ["frog", "sailboat", "cloud", "mushroom"]  # Edit before running, with at least two complete design families.
SPLIT_DIR = RUN_DIR / "split"
if not REVIEW_QUEUE.is_file():
    seed = REPO_DIR / "training" / "approved-seed.jsonl"
    if not seed.is_file():
        raise FileNotFoundError(f"Approved seed is missing: {seed}")
    shutil.copy2(seed, REVIEW_QUEUE)
    print("Copied 20 approved text-only seed rows to:", REVIEW_QUEUE)
    print("This small seed is for setup and preview; add more reviewed examples before expecting useful fine-tuning.")
if len(HOLDOUT_FAMILIES) < 2:
    raise ValueError("Choose at least two held-out families.")
train_jsonl, eval_jsonl = SPLIT_DIR / "train.jsonl", SPLIT_DIR / "eval.jsonl"
if train_jsonl.is_file() and eval_jsonl.is_file():
    print("Reusing saved split:", SPLIT_DIR)
elif SPLIT_DIR.exists():
    raise FileExistsError("Split folder is incomplete. Choose a new RUN_ID; do not overwrite data.")
else:
    subprocess.run([sys.executable, "training/prepare_dataset.py", "split", str(REVIEW_QUEUE),
                    "--output", str(SPLIT_DIR), "--holdout", *HOLDOUT_FAMILIES], check=True)

subprocess.run([sys.executable, "training/train_lora.py", "--train", str(train_jsonl),
                "--eval", str(eval_jsonl), "--check"], check=True)
OUTPUT_DIR = RUN_DIR / "adapter"
metadata = {"model": MODEL_ID, "repo_ref": REPO_REF, "gpu": gpu_name,
            "heldout_families": HOLDOUT_FAMILIES,
            "train_sha256": hashlib.sha256(train_jsonl.read_bytes()).hexdigest(),
            "eval_sha256": hashlib.sha256(eval_jsonl.read_bytes()).hexdigest()}
(RUN_DIR / "run-metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

base_output, base_summary = RUN_DIR / "heldout-base.jsonl", RUN_DIR / "heldout-base-summary.json"
if not base_summary.exists():
    subprocess.run([sys.executable, "training/evaluate_lora.py", "--data", str(eval_jsonl),
                    "--output", str(base_output), "--summary", str(base_summary),
                    "--model", MODEL_ID, "--image-max-side", "384", "--resume"], check=True)

training_log = RUN_DIR / "training.log"
command = [sys.executable, "training/train_lora.py", "--train", str(train_jsonl),
           "--eval", str(eval_jsonl), "--model", MODEL_ID, "--output", str(OUTPUT_DIR),
           "--image-max-side", "384", "--save-steps", "1", "--resume"]
with training_log.open("a") as log:
    subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)

adapter_output, adapter_summary = RUN_DIR / "heldout-adapter.jsonl", RUN_DIR / "heldout-adapter-summary.json"
if not adapter_summary.exists():
    subprocess.run([sys.executable, "training/evaluate_lora.py", "--data", str(eval_jsonl),
                    "--output", str(adapter_output), "--summary", str(adapter_summary),
                    "--model", MODEL_ID, "--adapter", str(OUTPUT_DIR),
                    "--image-max-side", "384", "--resume"], check=True)

print("Compare these files and review every output:")
print(base_summary, adapter_summary, base_output, adapter_output)
print("Training log:", training_log)
print("Adapter/checkpoints:", OUTPUT_DIR)


Copied 20 approved text-only seed rows to: /content/drive/MyDrive/Beadform-free-qwen/data/review-queue.jsonl
This small seed is for setup and preview; add more reviewed examples before expecting useful fine-tuning.
Compare these files and review every output:
/content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/heldout-base-summary.json /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/heldout-adapter-summary.json /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/heldout-base.jsonl /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/heldout-adapter.jsonl
Training log: /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/training.log
Adapter/checkpoints: /content/drive/MyDrive/Beadform-free-qwen/runs/pilot-01/adapter
